In [ ]:
%py
# PySpark script to insert a new record into the config_master table for ID region

from pyspark.sql.functions import lit

# Commenting out spark session initialization as it's assumed to be available in the Databricks environment
# spark = SparkSession.builder.appName("DatabricksTest").getOrCreate()

# Load the config_master table
config_master_df = spark.table("purgo_playground.config_master")

# Retrieve an existing record for transformation
existing_record_df = config_master_df.where(config_master_df.src_objt_name == "US_Sales").limit(1)

# Transform and append new record with specified values
new_record_df = existing_record_df.withColumn("src_objt_name", lit("ID_Sales")) \
                                  .withColumn("src_sys", lit("ID_Sales")) \
                                  .withColumn("f_format", lit("ID_MON_Sales_")) \
                                  .withColumn("s3_landing_path", lit("s3a://{s3_bucket}/landing/ID/ID_Sales/")) \
                                  .withColumn("s3_archive_path", lit("s3a://{s3_bucket}/archive/ID/ID_Sales/")) \
                                  .withColumn("country", lit("ID")) \
                                  .withColumn("region", lit("ID")) \
                                  .withColumn("affiliate_group", lit("ID")) \
                                  .withColumn("affiliate", lit("ID")) \
                                  .withColumn("src_layer", lit("ID_Sales")) \
                                  .withColumn("target_src_sys", lit("ID_Sales")) \
                                  .withColumn("delta_stg_tables", lit("stg_ID_sales")) \
                                  .withColumn("source_path", lit("/SecureFtp/-InternalX/ID/IN/DATA/Sales/")) \
                                  .withColumn("actual_file_name", lit('{"ID_MON_Sales_*": "stg_ID_wholesaler"}')) \
                                  .withColumn("dag_id", lit("LOAD_SALES_ID"))

# Ensure the config_id is unique
max_config_id = config_master_df.agg({"config_id": "max"}).collect()[0][0]
new_record_df = new_record_df.withColumn("config_id", lit(max_config_id + 1))

# Append the new record to the config_master table
new_record_df.write.format("delta").mode("append").saveAsTable("purgo_playground.config_master")

# Commenting out stopping the SparkSession in Databricks environment
# spark.stop()